## Installing NLTK and Spacy

In [1]:
!pip install nltk spacy docx2txt pandas matplotlib wordcloud

In [2]:
import nltk

In [3]:
nltk.download('punkt')        # For tokenization
nltk.download('stopwords')    # For stop words
nltk.download('wordnet')      # For lemmatization
nltk.download('averaged_perceptron_tagger') # For POS tagging
nltk.download('omw-1.4')      # For WordNet data
nltk.download('punkt_tab')    # Download the missing resource for sentence tokenization

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
# Download spaCy model
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 106.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Data Loading

In a real-world scenario, text data can come from various sources like text files (.txt), PDFs (.pdf), CSVs (.csv), Word documents (.docx), databases, or web scraping.

For this notebook, we will primarily work with a sample text string to demonstrate the preprocessing steps clearly. However, let's briefly touch upon loading from different formats.

* **.txt files:** Standard file reading (`open('filename.txt', 'r').read()`).
* **.csv files:** Using pandas (`pandas.read_csv('filename.csv')`).
* **.pdf files:** Requires libraries like `PyPDF2` or `pdfminer.six`.
* **.docx files:** Requires libraries like `python-docx` or `docx2txt`.
* **Databases/Web:** Requires database connectors (e.g., `psycopg2`, `sqlite3`) or web scraping libraries (`requests`, `BeautifulSoup`).


In [5]:
# Sample text representing a part of a document
sample_document = """
Machine learning (ML) is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data. Algorithms build a mathematical model based on sample data, known as "training data", in order to make predictions or decisions without being explicitly programmed to perform the task.

Machine learning algorithms are used in a wide variety of applications, such as email filtering and computer vision, where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks. A subset of machine learning is closely related to computational statistics, which focuses on making predictions using computers. Mathematical optimization delivers methods, theory and application domains to the field of machine learning. Data mining is a related field of study, focusing on exploratory data analysis through unsupervised learning.

In its application across business problems, machine learning is also referred to as predictive analytics.
"""

In [6]:
# Let's also add some text with punctuation, numbers, and mixed casing
noisy_text = """
Machine learning rocks! It's revolutionizing the world in 2023 (and beyond!).
Visit our site: http://example.com for more info.
This is awesome!!! We collected 1,234 data points.
Softbank and Google are major players.
"""

In [7]:
print("--- Sample Document ---")
print(sample_document)
print("\n--- Noisy Text ---")
print(noisy_text)

--- Sample Document ---

Machine learning (ML) is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data. Algorithms build a mathematical model based on sample data, known as "training data", in order to make predictions or decisions without being explicitly programmed to perform the task.

Machine learning algorithms are used in a wide variety of applications, such as email filtering and computer vision, where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks. A subset of machine learning is closely related to computational statistics, which focuses on making predictions using computers. Mathematical optimization delivers methods, theory and application domains to the field of machine learning. Data mining is a related field of study, focusing on exploratory data analysis through unsupervised learning.

In its application across business problems, mach

## Initial Text Cleaning
Raw text often contains noise like punctuation, numbers, special characters, URLs, and inconsistent casing. Cleaning involves removing or standardizing these elements.

We'll combine the sample texts for cleaning demonstration.

In [8]:
import re
import string

# Combine texts
raw_text = sample_document + "\n" + noisy_text

print("Original Raw Text Length:", len(raw_text))
print("--- --------------------------- ---")
print("--- Original Raw Text ---")
print(raw_text)

Original Raw Text Length: 1254
--- --------------------------- ---
--- Original Raw Text ---

Machine learning (ML) is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data. Algorithms build a mathematical model based on sample data, known as "training data", in order to make predictions or decisions without being explicitly programmed to perform the task.

Machine learning algorithms are used in a wide variety of applications, such as email filtering and computer vision, where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks. A subset of machine learning is closely related to computational statistics, which focuses on making predictions using computers. Mathematical optimization delivers methods, theory and application domains to the field of machine learning. Data mining is a related field of study, focusing on exploratory data analysis through unsu

### 1. Convert to Lowercase

In [9]:
cleaned_text = raw_text.lower()

print("\n--- After Lowercasing ---")
print(cleaned_text)


--- After Lowercasing ---

machine learning (ml) is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data. algorithms build a mathematical model based on sample data, known as "training data", in order to make predictions or decisions without being explicitly programmed to perform the task.

machine learning algorithms are used in a wide variety of applications, such as email filtering and computer vision, where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks. a subset of machine learning is closely related to computational statistics, which focuses on making predictions using computers. mathematical optimization delivers methods, theory and application domains to the field of machine learning. data mining is a related field of study, focusing on exploratory data analysis through unsupervised learning.

in its application across business problems, m

### 2. Remove URLs (using regex)

#### Use the following prompt in GPT

* Write a Python regular expression that matches
   URLs starting with "http://" or "https://", followed by one or more characters that can include letters,
   digits, special characters like $-_@.&+, !*(),, or percent-encoded characters like %20. Use re.compile() to compile the pattern.



#### url_pattern = re.compile(
    r'http[s]?://'  # Match "http://" or "https://"
    r'(?:'
        r'[a-zA-Z]'        # Match any letter (uppercase or lowercase)
        r'|'              
        r'[0-9]'           # Match any digit
        r'|'              
        r'[$-_@.&+]'       # Match any of the listed special characters
        r'|'              
        r'[!*\\(\\),]'     # Match other allowed characters (escaped)
        r'|'              
        r'(?:%[0-9a-fA-F][0-9a-fA-F])'  # Match percent-encoded characters
    r')+'
####)

In [10]:

url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
  #For now we can generate this regex using ChatGPT else you can use this link for reference https://regex101.com/library/OY96XI


cleaned_text = url_pattern.sub(r'', cleaned_text)
print("\n--- After Removing URLs ---")
print(cleaned_text)


--- After Removing URLs ---

machine learning (ml) is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data. algorithms build a mathematical model based on sample data, known as "training data", in order to make predictions or decisions without being explicitly programmed to perform the task.

machine learning algorithms are used in a wide variety of applications, such as email filtering and computer vision, where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks. a subset of machine learning is closely related to computational statistics, which focuses on making predictions using computers. mathematical optimization delivers methods, theory and application domains to the field of machine learning. data mining is a related field of study, focusing on exploratory data analysis through unsupervised learning.

in its application across business problems,

### 3. Remove punctuation (using string.punctuation and translate)

In [11]:

# string.punctuation includes !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

translator = str.maketrans('', '', string.punctuation)

# This line creates a translation table that tells Python which characters to delete from a string.

# str.maketrans(from, to, delete) creates a mapping for str.translate().

#In this case:

# from = '' and to = '': no character substitutions.

# delete = string.punctuation: remove all punctuation characters.

cleaned_text = cleaned_text.translate(translator)
print("\n--- After Removing Punctuation ---")
print(cleaned_text)


--- After Removing Punctuation ---

machine learning ml is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data algorithms build a mathematical model based on sample data known as training data in order to make predictions or decisions without being explicitly programmed to perform the task

machine learning algorithms are used in a wide variety of applications such as email filtering and computer vision where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks a subset of machine learning is closely related to computational statistics which focuses on making predictions using computers mathematical optimization delivers methods theory and application domains to the field of machine learning data mining is a related field of study focusing on exploratory data analysis through unsupervised learning

in its application across business problems machine le

### 4. Remove numbers (using regex)
### Removes digits (0-9)

In [12]:
cleaned_text = re.sub(r'\d+', '', cleaned_text)
print("\n--- After Removing Numbers ---")
print(cleaned_text)


--- After Removing Numbers ---

machine learning ml is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data algorithms build a mathematical model based on sample data known as training data in order to make predictions or decisions without being explicitly programmed to perform the task

machine learning algorithms are used in a wide variety of applications such as email filtering and computer vision where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks a subset of machine learning is closely related to computational statistics which focuses on making predictions using computers mathematical optimization delivers methods theory and application domains to the field of machine learning data mining is a related field of study focusing on exploratory data analysis through unsupervised learning

in its application across business problems machine learni

### 5. Remove extra whitespace (including newlines and multiple spaces)
### Replace multiple spaces with a single space

In [13]:
cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
print("\n--- After Removing Extra Whitespace ---")
print(cleaned_text)


--- After Removing Extra Whitespace ---
machine learning ml is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data algorithms build a mathematical model based on sample data known as training data in order to make predictions or decisions without being explicitly programmed to perform the task machine learning algorithms are used in a wide variety of applications such as email filtering and computer vision where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks a subset of machine learning is closely related to computational statistics which focuses on making predictions using computers mathematical optimization delivers methods theory and application domains to the field of machine learning data mining is a related field of study focusing on exploratory data analysis through unsupervised learning in its application across business problems machine 

In [14]:
print("\nCleaned Text Length:", len(cleaned_text))

# Note: Removing non-English words is more complex and often involves language detection libraries or checking words against a dictionary.



Cleaned Text Length: 1186


# 5. Tokenization
**Tokenization** is the process of breaking down a text into individual units called tokens. These tokens can be words, sentences, or even sub-word units.

**5.1 Sentence Tokenization**
Splitting text into sentences. Useful for tasks that require sentence-level analysis.

In [15]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 104.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [16]:
from nltk.tokenize import sent_tokenize
import spacy

# Load the small English spaCy model
nlp = spacy.load('en_core_web_sm')


In [17]:
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")

In [19]:
doc[0]

Apple

In [20]:
doc[0].pos_

'PROPN'

In [26]:
doc[1].pos_

'AUX'

In [21]:
# Using NLTK
nltk_sentences = sent_tokenize(raw_text) # Using raw_text to show sentence boundary handling
print("--- Sentence Tokenization (NLTK) ---")
for i, sentence in enumerate(nltk_sentences):
    print(f"Sentence {i+1}: {sentence}")

--- Sentence Tokenization (NLTK) ---
Sentence 1: 
Machine learning (ML) is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data.
Sentence 2: Algorithms build a mathematical model based on sample data, known as "training data", in order to make predictions or decisions without being explicitly programmed to perform the task.
Sentence 3: Machine learning algorithms are used in a wide variety of applications, such as email filtering and computer vision, where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks.
Sentence 4: A subset of machine learning is closely related to computational statistics, which focuses on making predictions using computers.
Sentence 5: Mathematical optimization delivers methods, theory and application domains to the field of machine learning.
Sentence 6: Data mining is a related field of study, focusing on exploratory data analys

In [22]:
print(raw_text)


Machine learning (ML) is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data. Algorithms build a mathematical model based on sample data, known as "training data", in order to make predictions or decisions without being explicitly programmed to perform the task.

Machine learning algorithms are used in a wide variety of applications, such as email filtering and computer vision, where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks. A subset of machine learning is closely related to computational statistics, which focuses on making predictions using computers. Mathematical optimization delivers methods, theory and application domains to the field of machine learning. Data mining is a related field of study, focusing on exploratory data analysis through unsupervised learning.

In its application across business problems, machine learning is also ref

In [23]:
# Using spaCy
# spaCy processes text as a Doc object, which has sentences built-in
spacy_doc_raw = nlp(raw_text) # Using raw_text
spacy_sentences = [sent.text for sent in spacy_doc_raw.sents]
print("\n--- Sentence Tokenization (spaCy) ---")
for i, sentence in enumerate(spacy_sentences):
    print(f"Sentence {i+1}: {sentence}")


--- Sentence Tokenization (spaCy) ---
Sentence 1: 
Machine learning (ML) is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data.
Sentence 2: Algorithms build a mathematical model based on sample data, known as "training data", in order to make predictions or decisions without being explicitly programmed to perform the task.


Sentence 3: Machine learning algorithms are used in a wide variety of applications, such as email filtering and computer vision, where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks.
Sentence 4: A subset of machine learning is closely related to computational statistics, which focuses on making predictions using computers.
Sentence 5: Mathematical optimization delivers methods, theory and application domains to the field of machine learning.
Sentence 6: Data mining is a related field of study, focusing on exploratory data an

### 5.2 Word Tokenization
Splitting text into individual words or terms. This is often the first step after cleaning.

In [24]:
from nltk.tokenize import word_tokenize

# Using NLTK (on the cleaned text)
nltk_word_tokens = word_tokenize(cleaned_text)
print("--- Word Tokenization (NLTK) ---")
print(nltk_word_tokens[:20]) # Print first 20 tokens

--- Word Tokenization (NLTK) ---
['machine', 'learning', 'ml', 'is', 'a', 'field', 'of', 'study', 'in', 'artificial', 'intelligence', 'concerned', 'with', 'the', 'development', 'of', 'computer', 'algorithms', 'that', 'can']


In [28]:
nltk_word_tokens

['machine',
 'learning',
 'ml',
 'is',
 'a',
 'field',
 'of',
 'study',
 'in',
 'artificial',
 'intelligence',
 'concerned',
 'with',
 'the',
 'development',
 'of',
 'computer',
 'algorithms',
 'that',
 'can',
 'learn',
 'from',
 'and',
 'make',
 'predictions',
 'on',
 'data',
 'algorithms',
 'build',
 'a',
 'mathematical',
 'model',
 'based',
 'on',
 'sample',
 'data',
 'known',
 'as',
 'training',
 'data',
 'in',
 'order',
 'to',
 'make',
 'predictions',
 'or',
 'decisions',
 'without',
 'being',
 'explicitly',
 'programmed',
 'to',
 'perform',
 'the',
 'task',
 'machine',
 'learning',
 'algorithms',
 'are',
 'used',
 'in',
 'a',
 'wide',
 'variety',
 'of',
 'applications',
 'such',
 'as',
 'email',
 'filtering',
 'and',
 'computer',
 'vision',
 'where',
 'it',
 'is',
 'difficult',
 'or',
 'infeasible',
 'to',
 'develop',
 'conventional',
 'algorithms',
 'to',
 'perform',
 'the',
 'needed',
 'tasks',
 'a',
 'subset',
 'of',
 'machine',
 'learning',
 'is',
 'closely',
 'related',
 'to

In [27]:
# Using spaCy (on the cleaned text)
# spaCy's Doc object already contains tokens
spacy_doc_cleaned = nlp(cleaned_text)
spacy_word_tokens = [token.text for token in spacy_doc_cleaned]
print("\n--- Word Tokenization (spaCy) ---")
print(spacy_word_tokens[:20]) # Print first 20 tokens



--- Word Tokenization (spaCy) ---
['machine', 'learning', 'ml', 'is', 'a', 'field', 'of', 'study', 'in', 'artificial', 'intelligence', 'concerned', 'with', 'the', 'development', 'of', 'computer', 'algorithms', 'that', 'can']


# 6. Stop Word Removal
Stop words are common words (like 'the', 'is', 'in', 'and') that often do not carry significant meaning for analysis and can be removed to reduce noise and dimensionality.

In [29]:
from nltk.corpus import stopwords

# Get English stop words from NLTK
nltk_stop_words = set(stopwords.words('english'))
print("--- Sample NLTK Stop Words ---")
print(list(nltk_stop_words)[:10])

--- Sample NLTK Stop Words ---
["it'd", 'itself', 'on', 'too', 'mightn', "he'd", "she'd", "you've", 'between', 'if']


In [30]:
len(list(nltk_stop_words))

198

In [31]:
# Remove stop words using NLTK tokens
nltk_tokens_no_stopwords = [word for word in nltk_word_tokens if word not in nltk_stop_words]
print("\n--- NLTK Tokens after Stop Word Removal ---")
print(nltk_tokens_no_stopwords[:20])


--- NLTK Tokens after Stop Word Removal ---
['machine', 'learning', 'ml', 'field', 'study', 'artificial', 'intelligence', 'concerned', 'development', 'computer', 'algorithms', 'learn', 'make', 'predictions', 'data', 'algorithms', 'build', 'mathematical', 'model', 'based']


In [32]:
# SpaCy tokens have an `is_stop` attribute
spacy_tokens_no_stopwords = [token.text for token in spacy_doc_cleaned if not token.is_stop]
print("\n--- spaCy Tokens after Stop Word Removal ---")
print(spacy_tokens_no_stopwords[:20])


--- spaCy Tokens after Stop Word Removal ---
['machine', 'learning', 'ml', 'field', 'study', 'artificial', 'intelligence', 'concerned', 'development', 'computer', 'algorithms', 'learn', 'predictions', 'data', 'algorithms', 'build', 'mathematical', 'model', 'based', 'sample']


In [33]:
spacy_tokens_no_stopwords

['machine',
 'learning',
 'ml',
 'field',
 'study',
 'artificial',
 'intelligence',
 'concerned',
 'development',
 'computer',
 'algorithms',
 'learn',
 'predictions',
 'data',
 'algorithms',
 'build',
 'mathematical',
 'model',
 'based',
 'sample',
 'data',
 'known',
 'training',
 'data',
 'order',
 'predictions',
 'decisions',
 'explicitly',
 'programmed',
 'perform',
 'task',
 'machine',
 'learning',
 'algorithms',
 'wide',
 'variety',
 'applications',
 'email',
 'filtering',
 'computer',
 'vision',
 'difficult',
 'infeasible',
 'develop',
 'conventional',
 'algorithms',
 'perform',
 'needed',
 'tasks',
 'subset',
 'machine',
 'learning',
 'closely',
 'related',
 'computational',
 'statistics',
 'focuses',
 'making',
 'predictions',
 'computers',
 'mathematical',
 'optimization',
 'delivers',
 'methods',
 'theory',
 'application',
 'domains',
 'field',
 'machine',
 'learning',
 'data',
 'mining',
 'related',
 'field',
 'study',
 'focusing',
 'exploratory',
 'data',
 'analysis',
 'un

#7. Stemming and Lemmatization
These techniques reduce words to their root form to group together different inflections of the same word (e.g., "running", "runs", "ran" -> "run"). This helps in standardizing vocabulary.

**Stemming:** A heuristic process that cuts off the ends of words. It's faster but can result in non-dictionary words (e.g., "automate" from "automatically").

**Lemmatization:** A more sophisticated process that uses a vocabulary and morphological analysis to return the base or dictionary form of a word (known as the lemma). It's slower but produces actual words (e.g., "better" -> "good"). Lemmatization often requires the Part-of-Speech (POS) tag of the word to be accurate.

In [34]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
# nltk, spacy are already imported


# --- Stemming with NLTK (PorterStemmer) ---
porter_stemmer = PorterStemmer()
stemmed_tokens = [porter_stemmer.stem(word) for word in nltk_tokens_no_stopwords] # Apply to NLTK tokens after stop word removal

print("--- Stemmed Tokens (NLTK PorterStemmer) ---")
print(stemmed_tokens[:20])



--- Stemmed Tokens (NLTK PorterStemmer) ---
['machin', 'learn', 'ml', 'field', 'studi', 'artifici', 'intellig', 'concern', 'develop', 'comput', 'algorithm', 'learn', 'make', 'predict', 'data', 'algorithm', 'build', 'mathemat', 'model', 'base']


In [35]:

# --- Lemmatization with spaCy ---

# Extract lemmas from spaCy Doc object (on cleaned text, filtering stop words)
spacy_lemmatized_tokens = [token.lemma_ for token in spacy_doc_cleaned if not token.is_stop]

print("\n--- Lemmatized Tokens (spaCy) ---")
print(spacy_lemmatized_tokens[:20])

# Note: spaCy's lemmatizer is generally more accurate as it uses context and
# doesn't require manual POS tag conversion like NLTK's WordNetLemmatizer


--- Lemmatized Tokens (spaCy) ---
['machine', 'learning', 'ml', 'field', 'study', 'artificial', 'intelligence', 'concern', 'development', 'computer', 'algorithm', 'learn', 'prediction', 'datum', 'algorithm', 'build', 'mathematical', 'model', 'base', 'sample']


---

# ⚖️ Key Differences

| Feature              | Stemming 🪓            | Lemmatization 📚        |
|----------------------|------------------------|--------------------------|
| Approach             | Rule-based cutting     | Dictionary + grammar     |
| Accuracy             | Low                    | High                     |
| Speed                | Fast                   | Slower                   |
| Output quality       | Often incorrect        | Meaningful words         |
| Context awareness    | ❌ No                  | ✅ Yes (sometimes)       |

---

## 8. Part-of-Speech (POS) Tagging

POS tagging is the process of assigning a grammatical category (like noun, verb, adjective) to each word in a sentence. This is useful for understanding the structure and meaning of text and is often a precursor to other tasks like chunking or dependency parsing.


In [40]:
# Using spaCy (on the cleaned text Doc object)
print("\n--- POS Tagging with spaCy ---")
# spaCy tokens have .pos_ (coarse-grained) and .tag_ (fine-grained) attributes
for i, token in enumerate(spacy_doc_cleaned):
    print(f"{token.text}: {token.pos_} ({token.tag_})")
    if i >= 20: break # Limit output

# SpaCy's POS tagging is part of its standard pipeline and is generally very accurate.
# It provides both coarse-grained (`.pos_`) and fine-grained (`.tag_`) POS tags.


--- POS Tagging with spaCy ---
machine: NOUN (NN)
learning: NOUN (NN)
ml: PROPN (NNP)
is: AUX (VBZ)
a: DET (DT)
field: NOUN (NN)
of: ADP (IN)
study: NOUN (NN)
in: ADP (IN)
artificial: ADJ (JJ)
intelligence: NOUN (NN)
concerned: VERB (VBN)
with: ADP (IN)
the: DET (DT)
development: NOUN (NN)
of: ADP (IN)
computer: NOUN (NN)
algorithms: NOUN (NNS)
that: PRON (WDT)
can: AUX (MD)
learn: VERB (VB)


| Word       | pos_  | tag_ | Meaning                   |
| ---------- | ----- | ---- | ------------------------- |
| machine    | NOUN  | NN   | common noun               |
| ml         | PROPN | NNP  | proper noun (like a name) |
| is         | AUX   | VBZ  | auxiliary verb            |
| a          | DET   | DT   | determiner                |
| artificial | ADJ   | JJ   | adjective                 |


In [39]:
spacy_doc_cleaned

machine learning ml is a field of study in artificial intelligence concerned with the development of computer algorithms that can learn from and make predictions on data algorithms build a mathematical model based on sample data known as training data in order to make predictions or decisions without being explicitly programmed to perform the task machine learning algorithms are used in a wide variety of applications such as email filtering and computer vision where it is difficult or infeasible to develop conventional algorithms to perform the needed tasks a subset of machine learning is closely related to computational statistics which focuses on making predictions using computers mathematical optimization delivers methods theory and application domains to the field of machine learning data mining is a related field of study focusing on exploratory data analysis through unsupervised learning in its application across business problems machine learning is also referred to as predictiv

## UNET code.

<img src=https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/047/890/original/Screenshot_2023-09-12_at_10.37.24_AM.png?1694496498 width=800>

In [41]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# -----------------------------
# 🔧 Basic Building Block
# -----------------------------
def conv_block(x, filters):
    x = layers.Conv2D(filters, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    return x


# -----------------------------
# 🔽 Encoder Block
# -----------------------------
def encoder_block(x, filters):
    c = conv_block(x, filters)
    p = layers.MaxPooling2D((2, 2))(c)
    return c, p   # c = skip connection


# -----------------------------
# 🔼 Decoder Block
# -----------------------------
def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, (2, 2), strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip])   # Skip connection
    x = conv_block(x, filters)
    return x


# -----------------------------
# 🧱 Build U-Net
# -----------------------------
def build_unet(input_shape=(128, 128, 3)):
    inputs = layers.Input(input_shape)

    # 🔽 Encoder
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)

    # 🧠 Bottleneck
    b1 = conv_block(p4, 1024)

    # 🔼 Decoder
    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    # 🎯 Output layer
    outputs = layers.Conv2D(1, (1, 1), activation="sigmoid")(d4)

    model = Model(inputs, outputs, name="U-Net")
    return model


# -----------------------------
# 🚀 Create Model
# -----------------------------
model = build_unet()
model.summary()

Model: "U-Net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 128,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     36,928 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │    147,584 │ activation_2[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ activation_3[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │      1,024 │ conv2d_4[0][0]  

 Total params: 31,055,297 (118.47 MB)

 Trainable params: 31,043,521 (118.42 MB)

 Non-trainable params: 11,776 (46.00 KB)

## Doubts

In [42]:
import re

text = "I love NLP 😊 #AI"

# remove emojis & special chars
cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', text)
print(cleaned)

I love NLP  AI


In [43]:
re.sub(r'#\w+', '', text)

'I love NLP 😊 '

In [45]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.8 MB/s eta 0:00:00


In [46]:
import emoji

emoji.replace_emoji(text, replace='********')

'I love NLP ******** #AI'

In [47]:
w = ["APPLE", "Apple", "AppLE"]

In [50]:
for items in w:
  print(items.lower())

apple
apple
apple


In [52]:
import re
import emoji
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# -----------------------------
# 🔧 Individual Steps
# -----------------------------

def to_lower(text):
    return text.lower()

def remove_punctuation(text):
    return re.sub(r"[^\w\s]", "", text)

def normalize_repeated_chars(text):
    # loooove -> loove (keeps some emphasis but reduces noise)
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

def expand_contractions(text):
    contractions = {
        "can't": "cannot",
        "won't": "will not",
        "it's": "it is",
        "i'm": "i am",
        "don't": "do not"
    }
    for k, v in contractions.items():
        text = text.replace(k, v)
    return text

def handle_emojis(text, mode="remove"):
    if mode == "remove":
        return emoji.replace_emoji(text, replace="")
    elif mode == "text":
        return emoji.demojize(text)
    return text

def process_hashtags(text):
    # remove '#' but keep word
    return re.sub(r"#(\w+)", r"\1", text)

def lemmatize(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])

def remove_stopwords(text):
    doc = nlp(text)
    return " ".join([token.text for token in doc if not token.is_stop])


# -----------------------------
# 🚀 Full Pipeline
# -----------------------------
def normalize_text(text):
    print("Original:", text)

    text = to_lower(text)
    print("\nLowercased:", text)

    text = expand_contractions(text)
    print("\nAfter contraction expansion:", text)

    text = handle_emojis(text, mode="text")
    print("\nAfter emoji handling:", text)

    text = process_hashtags(text)
    print("\nAfter hashtag processing:", text)

    text = normalize_repeated_chars(text)
    print("\nAfter repeated char normalization:", text)

    text = remove_punctuation(text)
    print("\nAfter punctuation removal:", text)

    text = lemmatize(text)
    print("\nAfter lemmatization:", text)

    text = remove_stopwords(text)
    print("\nAfter stopword removal:", text)

    return text


# -----------------------------
# 🧪 Test Example
# -----------------------------
text = "OMG!!! I loooove Apple's new iPhone 15 Pro 😍😍!!! It's soooo good, can't believe it's only $999!!! #Apple #iPhone15"

final_text = normalize_text(text)

print("\nFinal Output:", final_text)

Original: OMG!!! I loooove Apple's new iPhone 15 Pro 😍😍!!! It's soooo good, can't believe it's only $999!!! #Apple #iPhone15

Lowercased: omg!!! i loooove apple's new iphone 15 pro 😍😍!!! it's soooo good, can't believe it's only $999!!! #apple #iphone15

After contraction expansion: omg!!! i loooove apple's new iphone 15 pro 😍😍!!! it is soooo good, cannot believe it is only $999!!! #apple #iphone15

After emoji handling: omg!!! i loooove apple's new iphone 15 pro :smiling_face_with_heart-eyes::smiling_face_with_heart-eyes:!!! it is soooo good, cannot believe it is only $999!!! #apple #iphone15

After hashtag processing: omg!!! i loooove apple's new iphone 15 pro :smiling_face_with_heart-eyes::smiling_face_with_heart-eyes:!!! it is soooo good, cannot believe it is only $999!!! apple iphone15

After repeated char normalization: omg!! i loove apple's new iphone 15 pro :smiling_face_with_heart-eyes::smiling_face_with_heart-eyes:!! it is soo good, cannot believe it is only $99!! apple iphone

In [54]:
from textblob import TextBlob

In [55]:
text = "I loove this product"
corrected = str(TextBlob(text).correct())

print(corrected)

I love this product
